# Exercise 2: CNN Architecture Evolution
## From LeNet (1998) to VGGNet (2014) — 16 Years of Progress in One Notebook

---

### Learning Objectives
- Implement LeNet-5, AlexNet (simplified), and VGGNet-style architectures in PyTorch
- Understand how architectural choices affect accuracy, parameters, and speed
- Visualize convolutional filters and feature maps to see **what the network actually learns**
- Appreciate why depth matters more than width in CNNs

### The Challenge
You will train 4 progressively deeper CNNs on **Fashion-MNIST** (10 clothing categories) and race them against each other.

---

## Google Colab Setup

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU` to enable GPU acceleration.
>
> Then run the **Setup** cell below, followed by **Run All** (`Runtime → Run all`).

In [ ]:
# PyTorch and torchvision are pre-installed in Colab GPU runtimes
import torch, torchvision
print(f'PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  — device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
plt.style.use('seaborn-v0_8-darkgrid')

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
# Data loading
transform_train = transforms.Compose([
    transforms.Resize(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
transform_test = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=256, shuffle=False, num_workers=2)

# Preview the dataset
images, labels = next(iter(testloader))
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for i in range(20):
    ax = axes[i // 10][i % 10]
    img = images[i].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Dataset Preview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 1: LeNet-5 (1998)

Yann LeCun's pioneering architecture for digit recognition. 60K parameters.

```
INPUT(32×32) → CONV(6,5×5) → AvgPool → CONV(16,5×5) → AvgPool → FC(120) → FC(84) → OUTPUT(10)
```

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),
            nn.Tanh(),
            nn.AvgPool2d(2, 2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 6 * 6, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

lenet = LeNet5()
params = sum(p.numel() for p in lenet.parameters())
print(f'LeNet-5 parameters: {params:,}')
print(lenet)

## Part 2: AlexNet (2012) — The ImageNet Revolution

Won ImageNet 2012 with 15.3% top-5 error (vs 26.2% runner-up). Key innovations: ReLU, Dropout, Data Augmentation.

We use a **scaled-down version** for 32×32 images.

```
INPUT(32×32) → CONV(64,3×3) → MaxPool → CONV(192,3×3) → CONV(384,3×3) → CONV(256,3×3) → FC(1024) → FC(512) → OUT(10)
```

In [ ]:
class AlexNetMini(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512), nn.ReLU(inplace=True),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

alexnet = AlexNetMini()
params = sum(p.numel() for p in alexnet.parameters())
print(f'AlexNet-Mini parameters: {params:,}')

## Part 3: VGGNet-Style (2014)

Karen Simonyan & Andrew Zisserman's insight: **use only 3×3 convolutions, but go very deep.** Two 3×3 convs have the same receptive field as one 5×5, but fewer parameters and more non-linearity.

In [ ]:
def vgg_block(in_channels, out_channels, num_convs):
    layers = []
    for _ in range(num_convs):
        layers += [nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)]
        in_channels = out_channels
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGGMini(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            vgg_block(1, 32, 2),   # 32×32 → 16×16
            vgg_block(32, 64, 2),  # 16×16 → 8×8
            vgg_block(64, 128, 3), # 8×8 → 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

vgg = VGGMini()
params = sum(p.numel() for p in vgg.parameters())
print(f'VGGMini parameters: {params:,}')

In [ ]:
def train_model(model, trainloader, testloader, epochs=10, name='Model'):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_losses, test_accs = [], []
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for X, y in trainloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in testloader:
                X, y = X.to(device), y.to(device)
                correct += (model(X).argmax(1) == y).sum().item()
                total += y.size(0)
        
        acc = correct / total
        train_losses.append(running_loss / len(trainloader))
        test_accs.append(acc)
        print(f'[{name}] Epoch {epoch+1:2d}/{epochs} | Loss: {running_loss/len(trainloader):.3f} | Test Acc: {acc:.1%}')
    
    elapsed = time.time() - start_time
    print(f'[{name}] Training time: {elapsed:.1f}s | Final accuracy: {test_accs[-1]:.1%}\n')
    return train_losses, test_accs, elapsed

# Train all models
results = {}
for name, model in [('LeNet-5', LeNet5()), ('AlexNet-Mini', AlexNetMini()), ('VGG-Mini', VGGMini())]:
    losses, accs, t = train_model(model, trainloader, testloader, epochs=10, name=name)
    results[name] = {'losses': losses, 'accs': accs, 'time': t, 
                     'params': sum(p.numel() for p in model.parameters()),
                     'model': model}

In [ ]:
# Architecture comparison dashboard
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

colors_arch = {'LeNet-5': '#e74c3c', 'AlexNet-Mini': '#f39c12', 'VGG-Mini': '#2ecc71'}

ax1 = fig.add_subplot(gs[0, :2])
for name, data in results.items():
    ax1.plot(range(1, 11), [a * 100 for a in data['accs']], 
             label=name, color=colors_arch[name], linewidth=2.5, marker='o', markersize=5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Test Accuracy Over Training', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11); ax1.set_ylim(50, 100)

ax2 = fig.add_subplot(gs[0, 2])
names = list(results.keys())
final_accs = [results[n]['accs'][-1] * 100 for n in names]
bars = ax2.bar(names, final_accs, color=[colors_arch[n] for n in names], edgecolor='white', linewidth=1.5)
for bar, acc in zip(bars, final_accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{acc:.1f}%', 
             ha='center', fontweight='bold')
ax2.set_title('Final Test Accuracy', fontweight='bold'); ax2.set_ylim(50, 100)

ax3 = fig.add_subplot(gs[1, 0])
param_counts = [results[n]['params'] / 1e3 for n in names]
bars = ax3.bar(names, param_counts, color=[colors_arch[n] for n in names], edgecolor='white')
for bar, p in zip(bars, param_counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{p:.0f}K', ha='center', fontweight='bold')
ax3.set_title('Parameters (thousands)', fontweight='bold')

ax4 = fig.add_subplot(gs[1, 1])
times = [results[n]['time'] for n in names]
bars = ax4.bar(names, times, color=[colors_arch[n] for n in names], edgecolor='white')
for bar, t in zip(bars, times):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{t:.0f}s', ha='center', fontweight='bold')
ax4.set_title('Training Time (seconds)', fontweight='bold')

ax5 = fig.add_subplot(gs[1, 2])
efficiency = [results[n]['accs'][-1] / (results[n]['params'] / 1e6) for n in names]
bars = ax5.bar(names, efficiency, color=[colors_arch[n] for n in names], edgecolor='white')
ax5.set_title('Accuracy per Million Params\n(Efficiency)', fontweight='bold')

plt.suptitle('CNN Architecture Comparison on Fashion-MNIST', fontsize=15, fontweight='bold', y=1.01)
plt.show()

## Part 4: What Does the Network See? — Filter & Feature Map Visualization

One of the most fascinating aspects of CNNs: we can **visualize exactly what each layer detects**.

In [ ]:
def visualize_feature_maps(model, image, layer_idx=0, model_name=''):
    model.eval()
    
    # Get activation hooks
    activations = {}
    hooks = []
    
    def get_activation(name):
        def hook(module, input, output):
            activations[name] = output.detach()
        return hook
    
    for i, layer in enumerate(model.features):
        hooks.append(layer.register_forward_hook(get_activation(f'layer_{i}')))
    
    with torch.no_grad():
        model(image.unsqueeze(0).to(device))
    
    for h in hooks:
        h.remove()
    
    # Get first conv layer activations
    key = list(activations.keys())[layer_idx]
    feat = activations[key][0].cpu().numpy()
    n_maps = min(16, feat.shape[0])
    
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    axes = axes.flatten()
    
    axes[0].imshow(image.squeeze().cpu().numpy(), cmap='gray')
    axes[0].set_title('Input Image', fontweight='bold')
    axes[0].axis('off')
    
    for i in range(1, n_maps):
        axes[i].imshow(feat[i-1], cmap='viridis')
        axes[i].set_title(f'Filter {i}', fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle(f'{model_name} — Feature Maps at {key}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Get a test image
test_img, test_label = testset[42]
print(f'Visualizing: {CLASS_NAMES[test_label]}')

# Visualize VGG feature maps at different depths
vgg_model = results['VGG-Mini']['model']
for layer_idx in [0, 3, 7]:
    visualize_feature_maps(vgg_model, test_img, layer_idx=layer_idx, model_name='VGG-Mini')

## Exercises

### Exercise A — ZF-Net Deconvolution (Visualization)
ZF-Net (Zeiler & Fergus, 2013) introduced **deconvolutions** to visualize what patterns maximally activate each filter. Implement Grad-CAM to highlight which image regions the network focuses on for classification:

1. Register a hook on the last convolutional layer
2. Compute gradients of the output class score w.r.t. the feature maps
3. Weight the feature maps by the mean gradient and overlay on the input image

### Exercise B — Receptive Field Calculation
For VGG-Mini, calculate the receptive field at each stage:
- After block 1 (2× 3×3 conv + pool)
- After block 2
- After block 3

Formula: $RF_l = RF_{l-1} + (k-1) \times \prod_{i=1}^{l-1} s_i$

### Exercise C — Transfer Learning vs. Training from Scratch
Load a pretrained ResNet18, modify its first conv layer to accept 1-channel input and fine-tune only the last layer. Compare against VGG-Mini trained from scratch.
```python
resnet = models.resnet18(pretrained=True)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet.fc = nn.Linear(512, 10)
# Freeze all layers except fc and conv1
```

### Discussion Questions
1. LeNet uses AvgPool; AlexNet and VGGNet use MaxPool. What is the theoretical difference? When would you prefer each?
2. VGGNet has 2× 3×3 convolutions where AlexNet has 1× 5×5. Show they have the same receptive field, but VGG has fewer parameters. How many?
3. Looking at the feature maps — what do early layers detect vs. deep layers?

---
## Exercise A: Grad-CAM — Where Does the Network Look?

**Technique**: Gradient-weighted Class Activation Mapping (Selvaraju et al., 2017)

### How it works
1. Forward-pass an image through the network and record the feature maps of the **last conv layer**
2. Back-propagate the score of the predicted class to get **gradients** at that layer
3. Average-pool the gradients across spatial dimensions → importance weights
4. Weighted-sum the feature maps → raw heatmap; apply ReLU + normalise
5. Upsample to input size and overlay on the image

### Input
```
model  : trained VGG-Mini (already in `results['VGG-Mini']['model']`)
image  : a single test image tensor  shape (1, 32, 32)
```

### Expected Output
A 3-panel figure:
```
┌──────────────┬──────────────┬──────────────────────────────┐
│ Input image  │  Grad-CAM    │ Overlay (heatmap on image)   │
│ (greyscale)  │  heatmap     │ red = most attended region   │
└──────────────┴──────────────┴──────────────────────────────┘
```
For a **Sandal** the network should highlight the sole/strap area.
For a **Pullover** it should highlight the body/sleeves.

**Fill in the three `# YOUR CODE HERE` sections below.**

In [ ]:
import torch.nn.functional as F
from matplotlib.colors import LinearSegmentedColormap

def compute_gradcam(model, image, target_class=None):
    """
    Returns
    -------
    heatmap       : np.ndarray  shape (H, W), values in [0, 1]
    predicted_cls : int
    """
    model.eval()
    gradients, activations = [], []

    def save_grad(grad):       gradients.append(grad)
    def fwd_hook(m, inp, out):
        activations.append(out)
        out.register_hook(save_grad)

    # ── Step 1: find the last Conv2d in model.features ─────────────────────
    # YOUR CODE HERE
    last_conv = None
    for layer in model.features.modules():
        if isinstance(layer, nn.Conv2d):
            last_conv = layer          # keeps overwriting → ends on the last one
    hook = last_conv.register_forward_hook(fwd_hook)

    # ── Step 2: forward pass ────────────────────────────────────────────────
    img_t = image.unsqueeze(0).to(device)
    output = model(img_t)
    if target_class is None:
        target_class = output.argmax(1).item()

    # ── Step 3: backprop class score → gradients at last conv layer ─────────
    # YOUR CODE HERE
    model.zero_grad()
    output[0, target_class].backward()

    # ── Step 4: Grad-CAM formula ────────────────────────────────────────────
    # YOUR CODE HERE
    grads = gradients[0]                                    # (1, C, H, W)
    acts  = activations[0].detach()                         # (1, C, H, W)
    weights = grads.mean(dim=[2, 3], keepdim=True)          # global avg pool → (1, C, 1, 1)
    cam = (weights * acts).sum(dim=1, keepdim=True)         # weighted sum over channels
    cam = F.relu(cam)                                       # only positive contributions
    cam = cam.squeeze().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # normalise to [0,1]

    hook.remove()
    return cam, target_class


def show_gradcam(model, dataset, indices, model_name='VGG-Mini'):
    """Visualise Grad-CAM for a list of image indices."""
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1: axes = axes[None, :]     # ensure 2-D indexing

    cmap_heat = LinearSegmentedColormap.from_list('redblue',
                    ['#2166ac', '#fdae61', '#d73027'], N=256)

    for row, idx in enumerate(indices):
        img, label = dataset[idx]
        heatmap, pred = compute_gradcam(model, img)

        # Upsample heatmap to 32×32
        hmap_up = F.interpolate(
            torch.tensor(heatmap)[None, None],
            size=(32, 32), mode='bilinear', align_corners=False
        ).squeeze().numpy()

        raw = img.squeeze().cpu().numpy()           # (32,32)
        rgb = np.stack([raw]*3, axis=-1)            # (32,32,3) greyscale→RGB
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

        # Panel 1: input
        axes[row, 0].imshow(raw, cmap='gray')
        axes[row, 0].set_title(f'Input: {CLASS_NAMES[label]}', fontweight='bold')
        axes[row, 0].axis('off')

        # Panel 2: raw heatmap
        axes[row, 1].imshow(hmap_up, cmap=cmap_heat)
        axes[row, 1].set_title(f'Grad-CAM\npred={CLASS_NAMES[pred]}', fontsize=9)
        axes[row, 1].axis('off')

        # Panel 3: overlay
        overlay = 0.5 * rgb + 0.5 * plt.cm.jet(hmap_up)[:, :, :3]
        overlay = np.clip(overlay, 0, 1)
        axes[row, 2].imshow(overlay)
        correct = '✓' if pred == label else '✗'
        axes[row, 2].set_title(f'Overlay  {correct}', fontsize=10)
        axes[row, 2].axis('off')

    plt.suptitle(f'{model_name} — Grad-CAM Attention Maps', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── Run on 5 diverse test images ───────────────────────────────────────────
vgg_model = results['VGG-Mini']['model'].to(device)
# Pick one example per class for a balanced view
class_indices = {}
for i, (_, lbl) in enumerate(testset):
    if lbl not in class_indices and len(class_indices) < 5:
        class_indices[lbl] = i
    if len(class_indices) == 5:
        break

show_gradcam(vgg_model, testset, list(class_indices.values()), model_name='VGG-Mini')

---
## Exercise B: Receptive Field Calculation

The **receptive field (RF)** of a neuron is the region of the input image that can influence its activation.
Deeper neurons integrate information from larger image regions.

### Formula (stride-accumulating)
$$RF_l = RF_{l-1} + (k_l - 1) \cdot S_{l-1}$$
where $k_l$ is kernel size and $S_{l-1} = \prod_{i=1}^{l-1} s_i$ is the cumulative stride so far.

### Input — VGG-Mini layer sequence
| # | Layer | Kernel | Stride | RF after |
|---|-------|--------|--------|----------|
| 1 | Conv  | 3      | 1      | ?        |
| 2 | Conv  | 3      | 1      | ?        |
| 3 | Pool  | 2      | 2      | ?        |
| 4 | Conv  | 3      | 1      | ?        |
| 5 | Conv  | 3      | 1      | ?        |
| 6 | Pool  | 2      | 2      | ?        |
| 7 | Conv  | 3      | 1      | ?        |
| 8 | Conv  | 3      | 1      | ?        |
| 9 | Conv  | 3      | 1      | ?        |
|10 | Pool  | 2      | 2      | ?        |

### Expected Output
```
Layer  │ Kernel │ Stride │ Cum. Stride │ Receptive Field
────────┼────────┼────────┼─────────────┼────────────────
Conv 1  │   3    │   1    │      1      │       3
Conv 2  │   3    │   1    │      1      │       5
Pool 1  │   2    │   2    │      2      │       6
Conv 3  │   3    │   1    │      2      │      10
  ...                                   ...
Pool 3  │   2    │   2    │      8      │      ??
```
**After the final pool, the RF should be ≥ 28 × 28 px — the model sees most of the image.**

**Fill in the layer table below, then answer the two discussion questions.**

In [ ]:
# ── Receptive Field Calculator ──────────────────────────────────────────────
# Each entry: (layer_name, kernel_size, stride)
# YOUR CODE HERE: complete the table by filling in the correct k and s values

vgg_mini_layers = [
    # Block 1 (32→16)
    ('Conv-1',  3, 1),
    ('Conv-2',  3, 1),
    ('Pool-1',  2, 2),
    # Block 2 (16→8)
    ('Conv-3',  3, 1),
    ('Conv-4',  3, 1),
    ('Pool-2',  2, 2),
    # Block 3 (8→4)
    ('Conv-5',  3, 1),
    ('Conv-6',  3, 1),
    ('Conv-7',  3, 1),
    ('Pool-3',  2, 2),
]

def compute_receptive_fields(layer_spec):
    """Compute RF at each layer using the stride-accumulating formula."""
    rf = 1            # receptive field
    cum_stride = 1    # cumulative product of all strides so far
    rows = []
    for name, k, s in layer_spec:
        rf = rf + (k - 1) * cum_stride
        cum_stride *= s
        rows.append((name, k, s, cum_stride, rf))
    return rows

rows = compute_receptive_fields(vgg_mini_layers)

# ── Pretty-print the table ─────────────────────────────────────────────────
print(f'{'Layer':<10} {'Kernel':>7} {'Stride':>7} {'Cum.Stride':>12} {'Recept.Field':>14}')
print('─' * 55)
for name, k, s, cs, rf in rows:
    print(f'{name:<10} {k:>7} {s:>7} {cs:>12} {rf:>14}')

print(f'\nFinal receptive field: {rows[-1][-1]}×{rows[-1][-1]} px  '
      f'(input is 32×32 → model sees {min(100,round(rows[-1][-1]**2/32**2*100))}% of pixels)')

# ── Visualise the RF growth ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

layer_names = [r[0] for r in rows]
rf_sizes    = [r[4] for r in rows]

ax1.plot(range(len(rows)), rf_sizes, 'o-', color='#2ecc71', linewidth=2.5, markersize=8)
ax1.axhline(32, color='#e74c3c', linestyle='--', label='Input size (32 px)')
ax1.set_xticks(range(len(rows)))
ax1.set_xticklabels(layer_names, rotation=30, ha='right')
ax1.set_ylabel('Receptive Field (pixels)')
ax1.set_title('Receptive Field Growth Through VGG-Mini', fontweight='bold')
ax1.legend(); ax1.grid(True)

# ── Show RF squares overlaid on a 32×32 grid ──────────────────────────────
ax2.set_xlim(0, 32); ax2.set_ylim(0, 32)
ax2.set_aspect('equal'); ax2.invert_yaxis()
colors_rf = plt.cm.plasma(np.linspace(0.1, 0.9, len(rows)))
for i, (_, _, _, _, rf) in enumerate(rows):
    clipped = min(rf, 32)
    rect = plt.Rectangle((0, 0), clipped, clipped, linewidth=2,
                          edgecolor=colors_rf[i], facecolor='none',
                          label=f'{layer_names[i]} RF={rf}px')
    ax2.add_patch(rect)
ax2.set_title('Receptive Fields on 32×32 Input (clipped at edge)', fontweight='bold')
ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

# ── Discussion questions ────────────────────────────────────────────────────
print()
print('Discussion Q1: Two 3×3 convs give RF = 5. One 5×5 conv also gives RF = 5.')
print('  → VGG 2×(3×3) params: 2 × (3×3×C×C)     =', 2*9, '× C²')
print('  → Single (5×5) params:     (5×5×C×C)     =', 25, '× C²')
print(f'  → Savings: {(1-2*9/25)*100:.0f}% fewer weights + one extra ReLU nonlinearity!')

print()
print('Discussion Q2: AvgPool vs MaxPool')
print('  AvgPool  → smooth, preserves background texture (good for detection)')
print('  MaxPool  → keeps the strongest activation only (good for sharp features)')
print('  Modern nets often replace both with strided convolutions (ResNet, EfficientNet)')

---
## Exercise C: Transfer Learning vs. Training from Scratch

**Hypothesis**: A ResNet-18 pretrained on ImageNet (1.2 M images, 1 000 classes) should reach
higher accuracy on Fashion-MNIST *faster* than VGG-Mini trained from scratch,
even though the domains differ (colour photos vs greyscale clothing).

### Why does it work?
Early conv layers learn **universal features** (edges, textures, shapes) that transfer across tasks.
Only the final classifier head needs to be retrained.

### Strategy: Fine-tune only the head
| Layer group | Frozen? |
|-------------|--------|
| conv1 (adapted to 1-channel input) | **No** — must learn greyscale features |
| All other conv layers (layer1–layer4) | **Yes** — reuse ImageNet features |
| Fully-connected head | **No** — retrain for 10 Fashion-MNIST classes |

### Input
```
Pretrained ResNet-18 weights  (downloaded automatically)
Fashion-MNIST train/test loaders  (already loaded above)
```

### Expected Output
```
[ResNet-TL] Epoch  1/10 | Loss: 0.48 | Test Acc: 88.2%
  ...                                              ↑
[ResNet-TL] Epoch 10/10 | Loss: 0.28 | Test Acc: 92.1%

[VGG-Mini ] Epoch  1/10 | Loss: 0.71 | Test Acc: 79.6%
  ...                                              ↑ starts lower
[VGG-Mini ] Epoch 10/10 | Loss: 0.36 | Test Acc: 91.4%
```
Transfer learning typically wins *epoch 1* by a large margin, then they converge.

**Complete the two `# YOUR CODE HERE` sections.**

In [ ]:
import torchvision.models as models

# ── Build the fine-tuned ResNet ──────────────────────────────────────────────
def build_resnet_transfer(num_classes=10, freeze_body=True):
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # YOUR CODE HERE (Step 1): replace conv1 to accept 1-channel (greyscale) input
    # Original: Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    # Target  : Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    if freeze_body:
        # YOUR CODE HERE (Step 2): freeze ALL parameters, then unfreeze conv1 and fc
        for param in resnet.parameters():
            param.requires_grad = False
        for param in resnet.conv1.parameters():
            param.requires_grad = True

    # Replace final classifier for 10-class output
    resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
    return resnet

resnet_tl = build_resnet_transfer(freeze_body=True)
trainable = sum(p.numel() for p in resnet_tl.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet_tl.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  '
      f'({100*trainable/total:.1f}% of ResNet-18)')

# ── Train and compare ────────────────────────────────────────────────────────
# ResNet input must be ≥ 224×224; we resize Fashion-MNIST to 64×64 for speed
transform_tl = transforms.Compose([
    transforms.Resize(64),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
transform_tl_test = transforms.Compose([
    transforms.Resize(64), transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
tl_trainset = torchvision.datasets.FashionMNIST('./data', train=True,  download=False, transform=transform_tl)
tl_testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=False, transform=transform_tl_test)
tl_trainloader = torch.utils.data.DataLoader(tl_trainset, batch_size=128, shuffle=True,  num_workers=2)
tl_testloader  = torch.utils.data.DataLoader(tl_testset,  batch_size=128, shuffle=False, num_workers=2)

EPOCHS = 10
# Retrain VGG-Mini fresh for a fair comparison
_, vgg_accs_scratch, _ = train_model(VGGMini(), tl_trainloader, tl_testloader,
                                     epochs=EPOCHS, name='VGG-Mini (scratch)')
_, tl_accs, _          = train_model(resnet_tl,  tl_trainloader, tl_testloader,
                                     epochs=EPOCHS, name='ResNet-TL')

# ── Comparison plot ──────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = range(1, EPOCHS + 1)
ax1.plot(epochs_x, [a*100 for a in vgg_accs_scratch], 'o-', color='#e74c3c',
         linewidth=2.5, label='VGG-Mini (from scratch)')
ax1.plot(epochs_x, [a*100 for a in tl_accs], 's-', color='#2ecc71',
         linewidth=2.5, label='ResNet-18 (transfer learning)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Transfer Learning vs. Training from Scratch', fontweight='bold')
ax1.legend(fontsize=11); ax1.set_ylim(50, 100); ax1.grid(True)

# Epoch-1 advantage
advantage = (tl_accs[0] - vgg_accs_scratch[0]) * 100
ax1.annotate(f'Epoch-1 advantage\n+{advantage:.1f}% for TL',
             xy=(1, tl_accs[0]*100), xytext=(3, tl_accs[0]*100 - 8),
             arrowprops=dict(arrowstyle='->', color='gray'),
             fontsize=10, color='#2ecc71', fontweight='bold')

# Bar chart: final accuracy comparison
models_names = ['VGG-Mini\n(scratch)', 'ResNet-18\n(transfer)']
final_accs   = [vgg_accs_scratch[-1]*100, tl_accs[-1]*100]
bars = ax2.bar(models_names, final_accs, color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
for bar, acc in zip(bars, final_accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax2.set_title('Final Accuracy Comparison', fontweight='bold')
ax2.set_ylim(50, 100); ax2.grid(axis='y')

plt.tight_layout()
plt.show()

print(f'\nKey takeaway:')
print(f'  • VGG-Mini (scratch) final: {vgg_accs_scratch[-1]:.1%}')
print(f'  • ResNet-18  (TL)    final: {tl_accs[-1]:.1%}')
print(f'  • Transfer learning epoch-1 head-start: {advantage:+.1f}%')

---
## Final Summary: What You Learned

| Architecture | Year | Innovation | Params | Typical Acc |
|---|---|---|---|---|
| **LeNet-5**   | 1998 | First practical CNN | ~60 K   | ~87% |
| **AlexNet-Mini** | 2012 | ReLU, Dropout, GPU | ~4 M  | ~90% |
| **VGG-Mini**  | 2014 | Deep 3×3 stacking  | ~700 K  | ~91% |
| **ResNet-18** | 2015 | Residual connections | 11 M  | ~92% (TL) |

### Key Insights

1. **Depth > Width**: VGG-Mini has fewer parameters than AlexNet-Mini but beats it — stacking 3×3 convs is more effective than using larger kernels.
2. **Grad-CAM reveals focus**: the network genuinely learns to attend to the discriminative region (sole of a sandal, collar of a shirt), not random pixels.
3. **Receptive fields grow fast**: by the final pooling layer, every neuron effectively 'sees' the entire image — this is where global context lives.
4. **Transfer learning accelerates convergence**: even across domains (ImageNet photos → Fashion-MNIST), pretrained edge/texture detectors transfer well.

In [ ]:
# ── Per-class accuracy heatmap across all three architectures ───────────────
def per_class_accuracy(model, loader, n_classes=10):
    model.eval()
    correct = np.zeros(n_classes)
    total   = np.zeros(n_classes)
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            for c in range(n_classes):
                mask = y == c
                correct[c] += (preds[mask] == y[mask]).sum().item()
                total[c]   += mask.sum().item()
    return correct / (total + 1e-8)

arch_order = ['LeNet-5', 'AlexNet-Mini', 'VGG-Mini']
acc_matrix = np.zeros((len(arch_order), 10))
for i, name in enumerate(arch_order):
    acc_matrix[i] = per_class_accuracy(results[name]['model'], testloader)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(acc_matrix * 100, aspect='auto', cmap='RdYlGn', vmin=70, vmax=100)
ax.set_xticks(range(10)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(len(arch_order))); ax.set_yticklabels(arch_order, fontsize=11)
for i in range(len(arch_order)):
    for j in range(10):
        ax.text(j, i, f'{acc_matrix[i,j]*100:.0f}%',
                ha='center', va='center', fontsize=9,
                color='black' if acc_matrix[i, j] > 0.85 else 'white')
plt.colorbar(im, ax=ax, label='Accuracy (%)')
ax.set_title('Per-Class Accuracy Heatmap — All Architectures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Hardest and easiest classes per architecture
print('\nHardest class per architecture:')
for i, name in enumerate(arch_order):
    worst_cls = acc_matrix[i].argmin()
    print(f'  {name:<15}: {CLASS_NAMES[worst_cls]:<12} ({acc_matrix[i, worst_cls]:.1%})')

print('\nNote: Shirt vs T-shirt vs Pullover confusion is the classic Fashion-MNIST challenge.')